In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

TRACKS_PATH = (
    PROJECT_ROOT / "data" / "raw" / "tracks" / "tracks.csv"
)
FLEET_PATH = (
    PROJECT_ROOT / "data" / "raw" / "fleets" / "fleet.csv"
)
SPEED_FEATURES_PATH = (
    PROJECT_ROOT / "data" / "interim" / "speed_features.csv"
)

tracks = pd.read_csv(TRACKS_PATH)
fleet = pd.read_csv(FLEET_PATH)
speed_features = pd.read_csv(SPEED_FEATURES_PATH)

print("Tracks:", tracks.shape)
print("Fleet:", fleet.shape)
print("Speed features:", speed_features.shape)

Tracks: (151598, 19)
Fleet: (163, 5)
Speed features: (151598, 13)


In [3]:
df = tracks.merge(
    fleet,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_fleet")
)

df = df.merge(
    speed_features,
    on=["vehicle_id", "track_id"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_speed")
)

print("Merged dataset:", df.shape)
print("Missing vehicle specifications:",
      df["gross_vehicle_weight"].isna().sum())
print("Missing raw speed means:",
      df["raw_speed_mean"].isna().sum())

Merged dataset: (151598, 34)
Missing vehicle specifications: 25021
Missing raw speed means: 9


In [4]:
# 1. Check missing vehicle specifications
missing_specs = df[df["gross_vehicle_weight"].isna()]

print("Tracks with missing vehicle specifications:",
      len(missing_specs))
print("Unique affected vehicles:",
      missing_specs["vehicle_id"].nunique())
print("\nAffected vehicle IDs:")
print(sorted(missing_specs["vehicle_id"].unique()))

# 2. Inspect tracks with missing raw speed means
missing_speed = df[df["raw_speed_mean"].isna()]

speed_columns = [
    "vehicle_id",
    "track_id",
    "raw_observations",
    "telemetry_span_seconds",
    "raw_speed_mean",
    "raw_speed_min",
    "raw_speed_max",
    "missing_speed_count",
]

print("\nTracks with missing raw speed means:",
      len(missing_speed))
print(missing_speed[speed_columns].to_string(index=False))

# 3. Check whether the other raw speed statistics are missing too
print("\nMissing-value counts across speed features:")
print(
    df[
        [
            "raw_observations",
            "raw_speed_mean",
            "raw_speed_std",
            "raw_speed_min",
            "raw_speed_max",
        ]
    ].isna().sum()
)

Tracks with missing vehicle specifications: 25021
Unique affected vehicles: 14

Affected vehicle IDs:
[np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(125)]

Tracks with missing raw speed means: 9
 vehicle_id  track_id  raw_observations  telemetry_span_seconds  raw_speed_mean  raw_speed_min  raw_speed_max  missing_speed_count
         90    132379                 0                     NaN             NaN            NaN            NaN                    0
         90    132420                 0                     NaN             NaN            NaN            NaN                    0
        109    136589                 0                     NaN             NaN            NaN            NaN                    0
        109    136918                 0                     NaN             NaN            NaN            NaN                    0
     

In [7]:
# Work on a separate copy; preserve df and all original values.
features = df.copy()

# --------------------------------------------------
# A. TELEMETRY SUFFICIENCY
# --------------------------------------------------
features["insufficient_telemetry"] = (
    features["raw_observations"] < 3
)

# --------------------------------------------------
# B. DURATION VALIDITY
# --------------------------------------------------
features["start_time_parsed"] = pd.to_datetime(
    features["start_time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

features["stop_time_parsed"] = pd.to_datetime(
    features["stop_time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

features["duration_seconds"] = (
    features["stop_time_parsed"]
    - features["start_time_parsed"]
).dt.total_seconds()

features["duration_invalid"] = (
    features["duration_seconds"] < 0
)

# --------------------------------------------------
# C. SPEED SUMMARY FLAGS
# --------------------------------------------------
features["avg_speed_negative"] = (
    features["avg_speed"] < 0
)

features["avg_speed_gt_max_speed"] = (
    features["avg_speed"] > features["max_speed"]
)

features["raw_speed_missing"] = (
    features["raw_speed_mean"].isna()
)

# --------------------------------------------------
# D. SIGNAL-LOSS FLAGS
# --------------------------------------------------
features["n_signal_loss_negative"] = (
    features["n_signal_loss"] < 0
)

features["r_signal_loss_missing"] = (
    features["r_signal_loss"].isna()
)

features["r_signal_loss_out_of_range"] = (
    (features["r_signal_loss"] < 0)
    | (features["r_signal_loss"] > 1)
)

# --------------------------------------------------
# E. DISTANCE AND TRACK-GAP FLAGS
# --------------------------------------------------
features["distance_missing"] = (
    features["distance"].isna()
)

features["zero_distance"] = (
    features["distance"] == 0
)

features["zero_distance_speed_issue"] = (
    (features["distance"] == 0)
    & (features["max_speed"] > 1)
)

features["track_gap_missing"] = (
    features["track_gap"].isna()
)

# --------------------------------------------------
# F. REPORT FLAG COUNTS
# --------------------------------------------------
flag_columns = [
    "insufficient_telemetry",
    "duration_invalid",
    "avg_speed_negative",
    "avg_speed_gt_max_speed",
    "raw_speed_missing",
    "n_signal_loss_negative",
    "r_signal_loss_missing",
    "r_signal_loss_out_of_range",
    "distance_missing",
    "zero_distance",
    "zero_distance_speed_issue",
    "track_gap_missing",
]

print("Candidate quality-flag counts:")
print(features[flag_columns].sum().sort_values(ascending=False))

Candidate quality-flag counts:
avg_speed_gt_max_speed        45819
n_signal_loss_negative         1927
r_signal_loss_missing          1714
zero_distance                  1665
zero_distance_speed_issue       188
track_gap_missing               163
r_signal_loss_out_of_range       90
insufficient_telemetry           70
distance_missing                 49
duration_invalid                  9
raw_speed_missing                 9
avg_speed_negative                6
dtype: int64


In [8]:
# Inspect tracks marked as having insufficient telemetry
insufficient = features.loc[
    features["insufficient_telemetry"],
    [
        "vehicle_id",
        "track_id",
        "raw_observations",
        "raw_speed_mean",
        "distance",
        "avg_speed",
        "max_speed",
    ],
]

print("Flagged tracks:", len(insufficient))
print("\nObservation-count distribution:")
print(insufficient["raw_observations"].value_counts(dropna=False).sort_index())

print("\nTracks with missing raw speed mean:",
      insufficient["raw_speed_mean"].isna().sum())

print("\nTracks with missing distance:",
      insufficient["distance"].isna().sum())

display(insufficient.head(15))


Flagged tracks: 70

Observation-count distribution:
raw_observations
0     9
1     5
2    56
Name: count, dtype: int64

Tracks with missing raw speed mean: 9

Tracks with missing distance: 49


,vehicle_id,track_id,raw_observations,raw_speed_mean,distance,avg_speed,max_speed
85088,44,85088,2,0.025,NaN,NaN,NaN
85199,44,85199,2,0.015,NaN,NaN,NaN
85239,44,85239,2,0.015,NaN,NaN,NaN
85455,44,85455,2,0.025,NaN,NaN,NaN
85521,44,85521,2,0.020,NaN,NaN,NaN
85533,44,85533,2,0.020,0.0,0.0,0.03
85574,44,85574,2,0.035,NaN,NaN,NaN
85668,44,85668,2,0.040,0.0,0.0,0.06
85817,44,85817,2,0.040,NaN,NaN,NaN
86303,44,86303,2,0.060,NaN,NaN,NaN


In [9]:
# Fewer than 3 recorded telemetry observations
features["limited_telemetry"] = (
    features["raw_observations"] < 3
)

# Missing essential track measurements
features["insufficient_core_telemetry"] = (
    features["distance"].isna()
    | features["avg_speed"].isna()
    | features["max_speed"].isna()
)

In [30]:
# Work on the existing features dataframe.
# Preserve the original columns unchanged.

features["avg_speed_valid"] = features["avg_speed"].where(
    features["avg_speed"] >= 0
)

features["max_speed_valid"] = features["max_speed"].where(
    features["max_speed"] >= 0
)

features["n_signal_loss_valid"] = features["n_signal_loss"].where(
    features["n_signal_loss"] >= 0
)

features["r_signal_loss_valid"] = features["r_signal_loss"].where(
    features["r_signal_loss"].between(0, 1)
)

# Unit conversions
features["distance_km"] = features["distance"] / 1000

features["avg_speed_kmh"] = (
    features["avg_speed_valid"] * 3.6
)

features["max_speed_kmh"] = (
    features["max_speed_valid"] * 3.6
)

features["raw_speed_mean_kmh"] = (
    features["raw_speed_mean"] * 3.6
)

# Duration derived from timestamps
features["duration_minutes"] = (
    features["duration_seconds"] / 60
).where(features["duration_seconds"] >= 0)

# Verify the new columns
new_columns = [
    "avg_speed_valid",
    "max_speed_valid",
    "n_signal_loss_valid",
    "r_signal_loss_valid",
    "distance_km",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
    "duration_minutes",
]

display(features[new_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
avg_speed_valid,151543.0,3.621383,10.185694,0.000000,0.160000,0.590000,3.420000,1230.790000
max_speed_valid,151549.0,7.474009,9.809897,0.020000,0.130000,1.450000,15.210000,126.650000
n_signal_loss_valid,149622.0,1370.528672,9972.579609,0.000000,0.000000,1.000000,5.000000,532659.000000
r_signal_loss_valid,149794.0,0.009465,0.051525,0.000000,0.000000,0.000000,0.000035,0.999830
distance_km,151549.0,14.526373,42.081918,0.000000,0.020000,0.116000,2.058000,476.551000
avg_speed_kmh,151543.0,13.036978,36.668497,0.000000,0.576000,2.124000,12.312000,4430.844000
max_speed_kmh,151549.0,26.906431,35.315628,0.072000,0.468000,5.220000,54.756000,455.940000
raw_speed_mean_kmh,151589.0,12.339849,21.368028,0.033398,0.093719,0.529386,13.177681,340.224923
duration_minutes,151589.0,22.392300,42.244470,0.003350,1.090467,5.916250,20.620750,1029.733783


In [11]:
# Inspect the 20 highest track-level average speeds
speed_cols = [
    "vehicle_id",
    "track_id",
    "distance_km",
    "duration_minutes",
    "avg_speed",
    "max_speed",
    "raw_observations",
    "raw_speed_mean",
    "raw_speed_max",
    "avg_speed_gt_max_speed",
    "limited_telemetry",
]

display(
    features.nlargest(20, "avg_speed")[speed_cols]
)

# Compare track-level and raw telemetry speed summaries
print("Correlation between speed summaries:")
display(
    features[
        ["avg_speed", "max_speed", "raw_speed_mean", "raw_speed_max"]
    ].corr(method="spearman")
)

# Inspect the highest raw telemetry mean speeds
display(
    features.nlargest(20, "raw_speed_mean")[speed_cols]
)

,vehicle_id,track_id,distance_km,duration_minutes,avg_speed,max_speed,raw_observations,raw_speed_mean,raw_speed_max,avg_speed_gt_max_speed,limited_telemetry
138274,114,138274,277.421,3.756667,1230.79,2.33,2186,0.536395,2.33,True,False
49132,22,49132,182.083,2.706517,1121.26,9.15,1594,1.959812,9.15,True,False
135017,102,135017,68.080,1.085617,1045.18,6.86,646,1.956362,6.86,True,False
102170,54,102170,17.535,0.280250,1042.82,0.07,166,0.023193,0.07,True,False
93407,48,93407,104.576,2.277983,765.12,0.09,1342,0.022481,0.09,True,False
9707,5,9707,12.865,0.280383,764.73,0.07,166,0.025843,0.07,True,False
138899,118,138899,117.550,2.644717,740.79,6.78,1457,0.993473,6.78,True,False
91783,47,91783,226.810,5.702400,662.91,9.90,3358,3.061873,9.90,True,False
143352,135,143352,106.570,2.738200,648.66,13.56,1618,5.803560,13.56,True,False
150232,159,150232,88.133,2.312567,635.17,4.55,1375,1.051796,4.55,True,False


Correlation between speed summaries:


,avg_speed,max_speed,raw_speed_mean,raw_speed_max
avg_speed,1.000000,0.810304,0.856755,0.810222
max_speed,0.810304,1.000000,0.944445,0.999907
raw_speed_mean,0.856755,0.944445,1.000000,0.944202
raw_speed_max,0.810222,0.999907,0.944202,1.000000


,vehicle_id,track_id,distance_km,duration_minutes,avg_speed,max_speed,raw_observations,raw_speed_mean,raw_speed_max,avg_speed_gt_max_speed,limited_telemetry
42896,20,42896,0.000,1.123383,0.00,97.68,78,94.506923,94.95,False,False
49358,22,49358,0.000,11.396667,0.00,98.17,2737,94.336598,98.17,False,False
2775,2,2775,5.865,2.519133,38.80,84.82,76,70.896316,84.82,False,False
106166,54,106166,0.000,4.038717,0.00,70.51,29,70.010000,70.51,False,False
81401,40,81401,0.150,36.663800,0.07,72.77,41,69.216585,72.77,False,False
147501,150,147501,0.000,4.150733,0.00,70.27,37,67.959189,70.27,False,False
27588,14,27588,10.922,39.650367,4.59,70.95,667,67.148936,70.95,False,False
25431,13,25431,0.000,3.546767,0.00,71.57,24,65.588750,71.57,False,False
3851,2,3851,0.331,0.093483,59.01,69.25,48,62.398542,69.25,False,False
81376,40,81376,0.205,0.133817,25.53,75.69,6,58.183333,75.69,False,False


In [12]:
# Examine the distribution of positive average speeds
positive_avg_speed = features.loc[
    features["avg_speed"] > 0,
    "avg_speed"
]

display(
    positive_avg_speed.describe(
        percentiles=[
            0.50, 0.75, 0.90, 0.95, 0.99, 0.995
        ]
    ).to_frame("avg_speed_mps")
)

# Inspect the 30 records with the highest average speeds
high_speed_records = features.nlargest(
    30, "avg_speed"
).copy()

high_speed_records["avg_speed_kmh_check"] = (
    high_speed_records["avg_speed"] * 3.6
)

high_speed_records["max_speed_kmh_check"] = (
    high_speed_records["max_speed"] * 3.6
)

display(
    high_speed_records[
        [
            "vehicle_id",
            "track_id",
            "distance",
            "duration_seconds",
            "avg_speed",
            "max_speed",
            "raw_observations",
            "raw_speed_mean",
            "raw_speed_max",
            "avg_speed_gt_max_speed",
            "limited_telemetry",
        ]
    ]
)

,avg_speed_mps
count,149420.000000
mean,3.672836
std,10.248584
min,0.010000
50%,0.620000
75%,3.550000
90%,13.740000
95%,17.640000
99%,21.390000
99.5%,22.329050


,vehicle_id,track_id,distance,duration_seconds,avg_speed,max_speed,raw_observations,raw_speed_mean,raw_speed_max,avg_speed_gt_max_speed,limited_telemetry
138274,114,138274,277421.0,225.400,1230.79,2.33,2186,0.536395,2.33,True,False
49132,22,49132,182083.0,162.391,1121.26,9.15,1594,1.959812,9.15,True,False
135017,102,135017,68080.0,65.137,1045.18,6.86,646,1.956362,6.86,True,False
102170,54,102170,17535.0,16.815,1042.82,0.07,166,0.023193,0.07,True,False
93407,48,93407,104576.0,136.679,765.12,0.09,1342,0.022481,0.09,True,False
9707,5,9707,12865.0,16.823,764.73,0.07,166,0.025843,0.07,True,False
138899,118,138899,117550.0,158.683,740.79,6.78,1457,0.993473,6.78,True,False
91783,47,91783,226810.0,342.144,662.91,9.90,3358,3.061873,9.90,True,False
143352,135,143352,106570.0,164.292,648.66,13.56,1618,5.803560,13.56,True,False
150232,159,150232,88133.0,138.754,635.17,4.55,1375,1.051796,4.55,True,False


In [13]:
# Conservative threshold for investigating implausible truck speeds
AVG_SPEED_THRESHOLD_KMH = 120
AVG_SPEED_THRESHOLD_MPS = AVG_SPEED_THRESHOLD_KMH / 3.6

# Flag physically implausible reported average speeds
features["avg_speed_implausible"] = (
    features["avg_speed"] > AVG_SPEED_THRESHOLD_MPS
)

# Keep this separate from the speed-summary inconsistency flag
features["avg_speed_gt_max_speed"] = (
    features["avg_speed"] > features["max_speed"]
)

print(
    f"Average-speed threshold: "
    f"{AVG_SPEED_THRESHOLD_KMH} km/h "
    f"({AVG_SPEED_THRESHOLD_MPS:.2f} m/s)"
)

print(
    "Records above threshold:",
    features["avg_speed_implausible"].sum()
)

print(
    "Percentage of all records:",
    round(features["avg_speed_implausible"].mean() * 100, 3),
    "%"
)

print(
    "Records where average speed exceeds maximum speed:",
    features["avg_speed_gt_max_speed"].sum()
)

print(
    "Records satisfying both conditions:",
    (
        features["avg_speed_implausible"]
        & features["avg_speed_gt_max_speed"]
    ).sum()
)

Average-speed threshold: 120 km/h (33.33 m/s)
Records above threshold: 236
Percentage of all records: 0.156 %
Records where average speed exceeds maximum speed: 45819
Records satisfying both conditions: 234


In [14]:
# Inspect all records above the chosen average-speed threshold
high_speed_records = features.loc[
    features["avg_speed_implausible"]
].copy()

# Calculate distance / elapsed-time speed independently
high_speed_records["implied_speed_kmh"] = (
    high_speed_records["distance"]
    / high_speed_records["duration_seconds"]
    * 3.6
).where(
    high_speed_records["duration_seconds"] > 0
)

# Compare reported and independently implied speed
high_speed_records["speed_difference_kmh"] = (
    high_speed_records["avg_speed"] * 3.6
    - high_speed_records["implied_speed_kmh"]
)

display(
    high_speed_records[
        [
            "vehicle_id",
            "track_id",
            "distance",
            "duration_seconds",
            "avg_speed",
            "max_speed",
            "raw_observations",
            "raw_speed_mean",
            "raw_speed_max",
            "implied_speed_kmh",
            "avg_speed_gt_max_speed",
        ]
    ].sort_values("avg_speed", ascending=False)
)

,vehicle_id,track_id,distance,duration_seconds,avg_speed,max_speed,raw_observations,raw_speed_mean,raw_speed_max,implied_speed_kmh,avg_speed_gt_max_speed
138274,114,138274,277421.0,225.400,1230.79,2.33,2186,0.536395,2.33,4430.858917,True
49132,22,49132,182083.0,162.391,1121.26,9.15,1594,1.959812,9.15,4036.546360,True
135017,102,135017,68080.0,65.137,1045.18,6.86,646,1.956362,6.86,3762.654098,True
102170,54,102170,17535.0,16.815,1042.82,0.07,166,0.023193,0.07,3754.148082,True
93407,48,93407,104576.0,136.679,765.12,0.09,1342,0.022481,0.09,2754.436307,True
...,...,...,...,...,...,...,...,...,...,...,...
129424,76,129424,58402.0,1735.688,33.65,10.68,16148,1.679937,10.68,121.131908,True
120929,62,120929,29409.0,874.700,33.62,8.55,8566,0.618335,8.55,121.038527,True
109536,56,109536,17535.0,523.347,33.51,0.35,5123,0.029262,0.35,120.619780,True
137764,112,137764,86463.0,2581.185,33.50,25.75,25189,16.426957,25.75,120.590659,True


In [15]:
# Identify records above the 120 km/h average-speed threshold
high_speed_mask = features["avg_speed_implausible"]

# Compare high-speed records against all other records
high_speed_group = features.loc[high_speed_mask]
normal_speed_group = features.loc[~high_speed_mask]

hdop_cols = [
    "raw_hdop_mean",
    "raw_hdop_median",
    "raw_hdop_max",
]

print("High-average-speed records:", len(high_speed_group))
print("Other records:", len(normal_speed_group))

print("\nHDOP summary: high-average-speed records")
display(high_speed_group[hdop_cols].describe())

print("\nHDOP summary: other records")
display(normal_speed_group[hdop_cols].describe())

High-average-speed records: 236
Other records: 151362

HDOP summary: high-average-speed records


,raw_hdop_mean,raw_hdop_median,raw_hdop_max
count,236.000000,236.000000,236.000000
mean,1.315013,1.238559,5.510593
std,1.714602,1.677795,5.343894
min,0.428761,0.400000,0.700000
25%,0.678919,0.700000,1.700000
50%,0.922288,0.800000,2.800000
75%,1.271996,1.200000,8.025000
max,13.122917,14.300000,17.000000



HDOP summary: other records


,raw_hdop_mean,raw_hdop_median,raw_hdop_max
count,151353.000000,151353.000000,151353.000000
mean,1.157421,1.033635,5.055808
std,1.136199,1.027048,5.327049
min,0.200000,0.200000,0.200000
25%,0.605851,0.600000,1.300000
50%,0.813792,0.800000,2.400000
75%,1.271955,1.100000,7.000000
max,16.900000,16.900000,17.000000


In [16]:
# Separate extreme-average-speed records from all other records
high_speed_group = features.loc[
    features["avg_speed_implausible"]
].copy()

normal_speed_group = features.loc[
    ~features["avg_speed_implausible"]
].copy()

signal_cols = [
    "n_signal_loss",
    "d_signal_loss",
    "r_signal_loss",
]

print("Extreme-average-speed records:", len(high_speed_group))
print("Other records:", len(normal_speed_group))

print("\nSignal-loss summary: extreme-average-speed records")
display(high_speed_group[signal_cols].describe())

print("\nSignal-loss summary: other records")
display(normal_speed_group[signal_cols].describe())

Extreme-average-speed records: 236
Other records: 151362

Signal-loss summary: extreme-average-speed records


,n_signal_loss,d_signal_loss,r_signal_loss
count,236.000000,236.000000,236.000000
mean,-26998.487288,2157.728814,0.032380
std,28664.512689,6434.960215,0.125310
min,-126712.000000,0.000000,0.000000
25%,-42870.750000,1.000000,0.000017
50%,-25717.000000,94.000000,0.001865
75%,0.000000,873.750000,0.009054
max,45327.000000,43023.000000,0.999830



Signal-loss summary: other records


,n_signal_loss,d_signal_loss,r_signal_loss
count,151313.000000,151313.000000,149648.000000
mean,1164.753293,195.408722,0.041213
std,10371.455589,1367.569884,5.264797
min,-150494.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000
75%,5.000000,1.000000,0.000034
max,532659.000000,67936.000000,1495.000000


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
SPEED_ROOT = PROJECT_ROOT / "data" / "raw" / "speed"

# Select the 236 tracks already flagged by our 120 km/h threshold
extreme_tracks = features.loc[
    features["avg_speed_implausible"],
    ["vehicle_id", "track_id", "avg_speed", "max_speed",
     "distance", "duration_seconds", "n_signal_loss",
     "d_signal_loss", "r_signal_loss"]
].copy()

print("Extreme-speed tracks:", len(extreme_tracks))

Extreme-speed tracks: 236


In [18]:
def inspect_track_telemetry(vehicle_id, track_id):
    path = SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"

    if not path.exists():
        return {"vehicle_id": vehicle_id, "track_id": track_id,
                "file_found": False}

    raw = pd.read_csv(path)

    # Identify timestamp column
    timestamp_candidates = [
        c for c in raw.columns
        if c.lower() in {"epoch", "timestamp", "time", "datetime"}
    ]

    speed_candidates = [
        c for c in raw.columns
        if c.lower() in {"speed", "velocity"}
    ]

    if len(timestamp_candidates) != 1 or len(speed_candidates) != 1:
        raise ValueError(
            f"Cannot identify timestamp/speed columns in {path}. "
            f"Columns: {raw.columns.tolist()}"
        )

    timestamp_col = timestamp_candidates[0]
    speed_col = speed_candidates[0]

    ts = pd.to_numeric(raw[timestamp_col], errors="coerce")
    valid_ts = ts.dropna().sort_values()

    gaps = valid_ts.diff().dropna()

    return {
        "vehicle_id": vehicle_id,
        "track_id": track_id,
        "file_found": True,
        "raw_observations": len(raw),
        "valid_timestamps": len(valid_ts),
        "max_gap_seconds": gaps.max() if len(gaps) else np.nan,
        "median_gap_seconds": gaps.median() if len(gaps) else np.nan,
        "gaps_over_10s": (gaps > 10).sum(),
        "gaps_over_60s": (gaps > 60).sum(),
        "missing_speed_values": raw[speed_col].isna().sum(),
        "raw_speed_min": pd.to_numeric(
            raw[speed_col], errors="coerce"
        ).min(),
        "raw_speed_max": pd.to_numeric(
            raw[speed_col], errors="coerce"
        ).max(),
    }

gap_results = pd.DataFrame([
    inspect_track_telemetry(row.vehicle_id, row.track_id)
    for row in extreme_tracks.itertuples(index=False)
])

display(
    gap_results.sort_values(
        "max_gap_seconds", ascending=False
    ).head(30)
)

,vehicle_id,track_id,file_found,raw_observations,valid_timestamps,max_gap_seconds,median_gap_seconds,gaps_over_10s,gaps_over_60s,missing_speed_values,raw_speed_min,raw_speed_max
6,2,3896,True,72,72,145.048,0.102,1,1,0,0.01,58.07
3,2,2611,True,81,81,110.089,0.102,1,1,0,0.02,45.29
147,76,129424,True,16148,16148,102.653,0.101,1,1,0,0.00,10.68
212,147,146549,True,20893,20893,96.745,0.101,3,1,0,0.01,26.12
44,50,97306,True,55607,55607,89.383,0.102,1,1,0,0.00,25.85
4,2,2775,True,76,76,84.984,0.200,1,1,0,0.32,84.82
76,55,109011,True,46904,46904,74.506,0.102,13,2,0,0.00,26.10
77,55,109118,True,18557,18557,66.137,0.102,5,1,0,0.00,23.97
78,55,109198,True,12427,12427,61.041,0.102,3,1,0,0.00,25.23
57,54,104752,True,30398,30398,47.839,0.102,8,0,0,0.00,23.78


In [19]:
def inspect_largest_gap(vehicle_id, track_id):
    path = SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"

    raw = pd.read_csv(path)

    timestamp_col = next(
        c for c in raw.columns
        if c.lower() in {"epoch", "timestamp", "time", "datetime"}
    )
    speed_col = next(
        c for c in raw.columns
        if c.lower() in {"speed", "velocity"}
    )

    raw = raw.copy()
    raw["_timestamp"] = pd.to_numeric(
        raw[timestamp_col], errors="coerce"
    )
    raw["_speed"] = pd.to_numeric(
        raw[speed_col], errors="coerce"
    )

    raw = raw.dropna(subset=["_timestamp"]).sort_values("_timestamp")
    raw["_gap"] = raw["_timestamp"].diff()

    gap_idx = raw["_gap"].idxmax()
    position = raw.index.get_loc(gap_idx)

    # Return observations around the largest gap
    return raw.iloc[
        max(0, position - 3): min(len(raw), position + 4)
    ][[timestamp_col, speed_col, "_gap"]]


# Inspect the five tracks with the largest gaps
top_gap_tracks = gap_results.nlargest(5, "max_gap_seconds")

for row in top_gap_tracks.itertuples(index=False):
    print(
        f"\nVehicle {row.vehicle_id}, track {row.track_id}, "
        f"maximum gap {row.max_gap_seconds:.3f} seconds"
    )
    display(inspect_largest_gap(row.vehicle_id, row.track_id))


Vehicle 2, track 3896, maximum gap 145.048 seconds


,epoch,speed,_gap
1,1.641553e+09,57.95,0.102
2,1.641553e+09,57.80,0.102
3,1.641553e+09,50.38,6.396
4,1.641553e+09,1.01,145.048
5,1.641553e+09,1.02,0.201
6,1.641553e+09,1.14,0.200
7,1.641553e+09,1.45,0.200



Vehicle 2, track 2611, maximum gap 110.089 seconds


,epoch,speed,_gap
51,1.631380e+09,36.25,0.165
52,1.631380e+09,36.14,0.198
53,1.631380e+09,35.98,0.102
54,1.631380e+09,0.32,110.089
55,1.631380e+09,0.28,0.102
56,1.631380e+09,0.30,0.102
57,1.631380e+09,0.24,0.102



Vehicle 76, track 129424, maximum gap 102.653 seconds


,epoch,speed,_gap
15822,1.702914e+09,0.02,0.101
15823,1.702914e+09,0.04,0.101
15824,1.702914e+09,0.00,0.102
15825,1.702914e+09,0.83,102.653
15826,1.702914e+09,0.84,0.101
15827,1.702914e+09,0.84,0.101
15828,1.702914e+09,0.75,0.101



Vehicle 147, track 146549, maximum gap 96.745 seconds


,epoch,speed,_gap
231,1.695968e+09,17.43,0.101
232,1.695968e+09,17.40,0.101
233,1.695968e+09,17.41,0.101
234,1.695968e+09,11.20,96.745
235,1.695968e+09,11.16,0.101
236,1.695968e+09,11.12,0.101
237,1.695968e+09,11.11,0.101



Vehicle 50, track 97306, maximum gap 89.383 seconds


,epoch,speed,_gap
40099,1.644851e+09,17.27,0.102
40100,1.644851e+09,17.52,0.101
40101,1.644851e+09,17.75,0.101
40102,1.644851e+09,20.15,89.383
40103,1.644851e+09,20.15,0.102
40104,1.644851e+09,20.14,0.102
40105,1.644851e+09,20.17,0.102


In [20]:
extreme_tracks = features.loc[
    features["avg_speed_implausible"],
    [
        "vehicle_id",
        "track_id",
        "duration_seconds",
        "avg_speed",
        "distance",
        "telemetry_span_seconds",
    ],
].copy()

extreme_tracks["duration_minus_telemetry_span"] = (
    extreme_tracks["duration_seconds"]
    - extreme_tracks["telemetry_span_seconds"]
)

extreme_tracks["telemetry_coverage_ratio"] = (
    extreme_tracks["telemetry_span_seconds"]
    / extreme_tracks["duration_seconds"].where(
        extreme_tracks["duration_seconds"] > 0
    )
)

display(
    extreme_tracks[
        [
            "duration_seconds",
            "telemetry_span_seconds",
            "duration_minus_telemetry_span",
            "telemetry_coverage_ratio",
        ]
    ].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
)

display(
    extreme_tracks.sort_values(
        "telemetry_coverage_ratio",
        ascending=True,
    ).head(15)
)

,duration_seconds,telemetry_span_seconds,duration_minus_telemetry_span,telemetry_coverage_ratio
count,236.000000,236.000000,236.000000,236.000000
mean,1513.978945,1507.315373,6.663572,0.993648
std,1291.025872,1287.407197,56.717131,0.031807
min,0.613000,0.610000,0.001000,0.674405
25%,580.825750,579.474000,0.002750,0.999766
50%,1275.242000,1266.521500,0.004000,0.999996
75%,2061.248250,2061.247250,0.175500,0.999998
90%,3011.671500,2983.794000,3.467000,0.999999
max,7799.818000,7798.692000,847.655000,1.000000


,vehicle_id,track_id,duration_seconds,avg_speed,distance,telemetry_span_seconds,duration_minus_telemetry_span,telemetry_coverage_ratio
2775,2,2775,151.148,38.80,5865.0,101.935,49.213,0.674405
146549,147,146549,3106.575,38.45,119445.0,2258.920,847.655,0.727142
3851,2,3851,5.609,59.01,331.0,4.790,0.819,0.853985
142503,132,142503,1435.394,43.48,62418.0,1259.482,175.912,0.877447
3896,2,3896,179.070,61.46,11005.0,160.636,18.434,0.897057
144915,140,144915,409.024,172.13,70406.0,377.011,32.013,0.921733
138899,118,138899,158.683,740.79,117550.0,147.029,11.654,0.926558
124857,64,124857,56.763,34.62,1965.0,54.438,2.325,0.959040
122402,63,122402,128.656,224.80,28922.0,123.843,4.813,0.962590
144139,137,144139,1528.416,72.50,110807.0,1476.096,52.320,0.965768


In [21]:
import numpy as np
import pandas as pd

# Select the 236 extreme-average-speed tracks
extreme_tracks = features.loc[
    features["avg_speed_implausible"],
    [
        "vehicle_id",
        "track_id",
        "distance",
        "avg_speed",
        "duration_seconds",
        "d_signal_loss",
        "r_signal_loss",
    ],
].copy()

# Ensure numeric values
for col in [
    "distance",
    "avg_speed",
    "duration_seconds",
    "d_signal_loss",
    "r_signal_loss",
]:
    extreme_tracks[col] = pd.to_numeric(
        extreme_tracks[col], errors="coerce"
    )

# Calculate signal-loss distance as a fraction of total distance
extreme_tracks["signal_loss_distance_fraction"] = (
    extreme_tracks["d_signal_loss"]
    / extreme_tracks["distance"].where(extreme_tracks["distance"] > 0)
)

# Convert to percentage for easier interpretation
extreme_tracks["signal_loss_distance_pct"] = (
    extreme_tracks["signal_loss_distance_fraction"] * 100
)

# Inspect the most extreme average speeds
display(
    extreme_tracks.sort_values("avg_speed", ascending=False)
    .head(20)
    .round(4)
)

,vehicle_id,track_id,distance,avg_speed,duration_seconds,d_signal_loss,r_signal_loss,signal_loss_distance_fraction,signal_loss_distance_pct
138274,114,138274,277421.0,1230.79,225.400,0.0,0.0000,0.0000,0.0000
49132,22,49132,182083.0,1121.26,162.391,983.0,0.0054,0.0054,0.5399
135017,102,135017,68080.0,1045.18,65.137,94.0,0.0014,0.0014,0.1381
102170,54,102170,17535.0,1042.82,16.815,50.0,0.0029,0.0029,0.2851
93407,48,93407,104576.0,765.12,136.679,41.0,0.0004,0.0004,0.0392
9707,5,9707,12865.0,764.73,16.823,0.0,0.0000,0.0000,0.0000
138899,118,138899,117550.0,740.79,158.683,0.0,0.0000,0.0000,0.0000
91783,47,91783,226810.0,662.91,342.144,1525.0,0.0067,0.0067,0.6724
143352,135,143352,106570.0,648.66,164.292,2.0,0.0000,0.0000,0.0019
150232,159,150232,88133.0,635.17,138.754,0.0,0.0000,0.0000,0.0000


In [22]:
fraction = extreme_tracks["signal_loss_distance_fraction"]

print("Valid fraction records:", fraction.notna().sum())
print("Invalid or missing:", fraction.isna().sum())
print("Fraction above 1:", (fraction > 1).sum())

display(
    (fraction[fraction.between(0, 1)])
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .to_frame("signal_loss_distance_fraction")
)

print(
    "Tracks with signal-loss distance above 10% of total distance:",
    (fraction > 0.10).sum(),
)

Valid fraction records: 236
Invalid or missing: 0
Fraction above 1: 0


,signal_loss_distance_fraction
count,236.000000
mean,0.032380
std,0.125310
min,0.000000
25%,0.000017
50%,0.001865
75%,0.009054
90%,0.068496
95%,0.146200
max,0.999829


Tracks with signal-loss distance above 10% of total distance: 17


In [23]:
import numpy as np
import pandas as pd

def estimate_distance_from_speed(vehicle_id, track_id):
    path = SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"

    try:
        raw = pd.read_csv(path)
        epoch_col = next(
            c for c in raw.columns
            if c.lower() in {"epoch", "timestamp", "time", "datetime"}
        )
        speed_col = next(
            c for c in raw.columns
            if c.lower() in {"speed", "velocity"}
        )

        raw["_t"] = pd.to_numeric(raw[epoch_col], errors="coerce")
        raw["_v"] = pd.to_numeric(raw[speed_col], errors="coerce")
        raw = (
            raw.dropna(subset=["_t", "_v"])
            .sort_values("_t")
            .drop_duplicates("_t")
        )

        dt = raw["_t"].diff().to_numpy()
        speed = raw["_v"].to_numpy()

        # Integrate only across short gaps (at most 1 second).
        valid = (
            np.isfinite(dt)
            & (dt > 0)
            & (dt <= 1.0)
            & np.isfinite(speed)
        )

        # Trapezoidal integration: mean speed × elapsed time.
        estimated_distance = np.sum(
            ((speed[1:] + speed[:-1]) / 2)[valid[1:]]
            * dt[valid[1:]]
        )

        covered_seconds = dt[valid].sum()

        return {
            "vehicle_id": vehicle_id,
            "track_id": track_id,
            "raw_speed_integrated_distance_m": estimated_distance,
            "speed_integrated_seconds": covered_seconds,
            "raw_observations_used": int(valid.sum()),
        }

    except (OSError, ValueError, StopIteration, KeyError):
        return {
            "vehicle_id": vehicle_id,
            "track_id": track_id,
            "raw_speed_integrated_distance_m": np.nan,
            "speed_integrated_seconds": np.nan,
            "raw_observations_used": 0,
        }


# Run for all 236 extreme-speed tracks.
distance_checks = pd.DataFrame([
    estimate_distance_from_speed(row.vehicle_id, row.track_id)
    for row in extreme_tracks.itertuples(index=False)
])

distance_checks = distance_checks.merge(
    extreme_tracks[
        ["vehicle_id", "track_id", "distance",
         "avg_speed", "duration_seconds"]
    ],
    on=["vehicle_id", "track_id"],
    how="left",
)

distance_checks["recorded_to_integrated_ratio"] = (
    distance_checks["distance"]
    / distance_checks["raw_speed_integrated_distance_m"].replace(0, np.nan)
)

display(
    distance_checks.sort_values(
        "recorded_to_integrated_ratio",
        ascending=False,
    ).head(20).round(3)
)

IndexError: boolean index did not match indexed array along axis 0; size of axis is 8062 but size of corresponding boolean axis is 8061

In [24]:

def estimate_distance_from_speed(vehicle_id, track_id):
    path = SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"

    try:
        raw = pd.read_csv(path)

        epoch_col = next(
            c for c in raw.columns
            if c.lower() in {"epoch", "timestamp", "time", "datetime"}
        )
        speed_col = next(
            c for c in raw.columns
            if c.lower() in {"speed", "velocity"}
        )

        raw["_t"] = pd.to_numeric(raw[epoch_col], errors="coerce")
        raw["_v"] = pd.to_numeric(raw[speed_col], errors="coerce")

        raw = (
            raw.dropna(subset=["_t", "_v"])
            .sort_values("_t")
            .drop_duplicates("_t")
        )

        t = raw["_t"].to_numpy()
        v = raw["_v"].to_numpy()

        # One time interval between each consecutive pair of observations
        dt = np.diff(t)

        # Keep only short, positive intervals
        valid = (
            np.isfinite(dt)
            & (dt > 0)
            & (dt <= 1.0)
            & np.isfinite(v[:-1])
            & np.isfinite(v[1:])
        )

        # Trapezoidal integration over valid intervals
        interval_speed = (v[:-1] + v[1:]) / 2

        estimated_distance = np.sum(
            interval_speed[valid] * dt[valid]
        )

        covered_seconds = dt[valid].sum()

        return {
            "vehicle_id": vehicle_id,
            "track_id": track_id,
            "raw_speed_integrated_distance_m": estimated_distance,
            "speed_integrated_seconds": covered_seconds,
            "raw_observations_used": int(valid.sum()),
        }

    except (OSError, ValueError, StopIteration, KeyError):
        return {
            "vehicle_id": vehicle_id,
            "track_id": track_id,
            "raw_speed_integrated_distance_m": np.nan,
            "speed_integrated_seconds": np.nan,
            "raw_observations_used": 0,
        }


In [25]:
distance_checks = pd.DataFrame([
    estimate_distance_from_speed(row.vehicle_id, row.track_id)
    for row in extreme_tracks.itertuples(index=False)
])

distance_checks = distance_checks.merge(
    extreme_tracks[
        ["vehicle_id", "track_id", "distance",
         "avg_speed", "duration_seconds"]
    ],
    on=["vehicle_id", "track_id"],
    how="left",
)

distance_checks["recorded_to_integrated_ratio"] = (
    distance_checks["distance"]
    / distance_checks["raw_speed_integrated_distance_m"].replace(0, np.nan)
)

display(
    distance_checks.sort_values(
        "recorded_to_integrated_ratio",
        ascending=False,
    ).head(20).round(3)
)

,vehicle_id,track_id,raw_speed_integrated_distance_m,speed_integrated_seconds,raw_observations_used,distance,avg_speed,duration_seconds,recorded_to_integrated_ratio
45,54,102170,0.391,16.811,165,17535.0,1042.82,16.815,44880.983
40,48,93407,3.074,136.675,1341,104576.0,765.12,136.679,34019.518
9,5,9707,0.433,16.820,165,12865.0,764.73,16.823,29730.541
102,59,114748,7.039,307.910,3021,149795.0,486.49,307.913,21280.147
11,6,11767,15.303,633.125,6212,170415.0,267.86,636.205,11136.019
31,35,74467,17.247,676.318,6629,137462.0,195.52,703.058,7970.239
16,12,25036,4.798,196.618,1929,31717.0,161.31,196.622,6611.117
46,54,102213,45.139,2020.241,19821,268164.0,132.74,2020.244,5940.806
207,146,146128,10.973,228.729,2265,62017.0,271.14,228.730,5651.972
18,17,37770,6.241,299.321,2937,31964.0,106.79,299.324,5121.894


In [26]:
top_cases = distance_checks.nlargest(
    5, "recorded_to_integrated_ratio"
)

display(
    top_cases[
        [
            "vehicle_id",
            "track_id",
            "distance",
            "avg_speed",
            "duration_seconds",
            "raw_speed_integrated_distance_m",
            "speed_integrated_seconds",
            "recorded_to_integrated_ratio",
        ]
    ].round(4)
)

for row in top_cases.itertuples(index=False):
    path = SPEED_ROOT / str(row.vehicle_id) / f"{row.track_id}.csv"
    raw = pd.read_csv(path)

    speed_col = next(
        c for c in raw.columns
        if c.lower() in {"speed", "velocity"}
    )

    speed = pd.to_numeric(raw[speed_col], errors="coerce").dropna()

    print(
        f"\nVehicle {row.vehicle_id}, track {row.track_id}"
        f"\nRaw speed statistics (original units):"
    )
    print(speed.describe(percentiles=[0.5, 0.9, 0.99]))

,vehicle_id,track_id,distance,avg_speed,duration_seconds,raw_speed_integrated_distance_m,speed_integrated_seconds,recorded_to_integrated_ratio
45,54,102170,17535.0,1042.82,16.815,0.3907,16.811,44880.9831
40,48,93407,104576.0,765.12,136.679,3.0740,136.675,34019.5185
9,5,9707,12865.0,764.73,16.823,0.4327,16.820,29730.5407
102,59,114748,149795.0,486.49,307.913,7.0392,307.910,21280.1472
11,6,11767,170415.0,267.86,636.205,15.3030,633.125,11136.0190



Vehicle 54, track 102170
Raw speed statistics (original units):
count    166.000000
mean       0.023193
std        0.013531
min        0.000000
50%        0.020000
90%        0.040000
99%        0.060000
max        0.070000
Name: speed, dtype: float64

Vehicle 48, track 93407
Raw speed statistics (original units):
count    1342.000000
mean        0.022481
std         0.012023
min         0.000000
50%         0.020000
90%         0.040000
99%         0.055900
max         0.090000
Name: speed, dtype: float64

Vehicle 5, track 9707
Raw speed statistics (original units):
count    166.000000
mean       0.025843
std        0.012512
min        0.000000
50%        0.020000
90%        0.040000
99%        0.060000
max        0.070000
Name: speed, dtype: float64

Vehicle 59, track 114748
Raw speed statistics (original units):
count    3022.000000
mean        0.022862
std         0.027706
min         0.000000
50%         0.020000
90%         0.040000
99%         0.070000
max         0.950000
Name

In [27]:
top5 = distance_checks.nlargest(
    5, "recorded_to_integrated_ratio"
)[["vehicle_id", "track_id"]]

signal_fields = features[
    [
        "vehicle_id",
        "track_id",
        "distance",
        "avg_speed",
        "n_signal_loss",
        "d_signal_loss",
        "r_signal_loss",
    ]
]

top5_details = top5.merge(
    signal_fields,
    on=["vehicle_id", "track_id"],
    how="left",
)

top5_details["signal_loss_distance_fraction"] = (
    top5_details["d_signal_loss"]
    / top5_details["distance"].where(
        top5_details["distance"] > 0
    )
)

display(top5_details.round(5))

,vehicle_id,track_id,distance,avg_speed,n_signal_loss,d_signal_loss,r_signal_loss,signal_loss_distance_fraction
0,54,102170,17535.0,1042.82,-19062.0,50.0,0.00285,0.00285
1,48,93407,104576.0,765.12,-77615.0,41.0,0.00039,0.00039
2,5,9707,12865.0,764.73,0.0,0.0,0.00000,0.00000
3,59,114748,149795.0,486.49,-1226.0,0.0,0.00000,0.00000
4,6,11767,170415.0,267.86,-76395.0,716.0,0.00420,0.00420


In [28]:
features["speed_distance_validity"] = np.where(
    features["avg_speed"] > (120 / 3.6),
    "invalid",
    "valid"
)

In [32]:
n_loss = features["n_signal_loss_valid"].dropna()

q1 = n_loss.quantile(0.25)
q3 = n_loss.quantile(0.75)
upper_fence = q3 + 1.5 * (q3 - q1)

print("Upper fence:", upper_fence)
print("Observations above fence:", (n_loss > upper_fence).sum())
print("Percentage above fence:", (n_loss > upper_fence).mean() * 100)

display(
    features.loc[
        features["n_signal_loss_valid"] > upper_fence,
        [
            "vehicle_id",
            "track_id",
            "n_signal_loss_valid",
            "d_signal_loss",
            "r_signal_loss_valid",
            "distance_km",
            "duration_minutes",
        ],
    ]
    .sort_values("n_signal_loss_valid", ascending=False)
    .head(20)
)

Upper fence: 12.5
Observations above fence: 25499
Percentage above fence: 17.042279878627475


,vehicle_id,track_id,n_signal_loss_valid,d_signal_loss,r_signal_loss_valid,distance_km,duration_minutes
74784,35,74784,532659.0,246.0,0.012045,20.423,1029.733783
74708,35,74708,391263.0,3.0,0.000070,42.811,736.253283
63263,28,63263,211555.0,0.0,0.000000,213.189,363.094333
63830,29,63830,209251.0,0.0,0.000000,0.004,355.796783
52439,24,52439,205715.0,0.0,0.000000,249.634,351.395000
54541,25,54541,203732.0,0.0,0.000000,185.510,349.324100
130715,80,130715,200072.0,0.0,0.000000,0.578,352.143583
53439,24,53439,197234.0,0.0,0.000000,241.239,342.807500
139732,121,139732,195357.0,0.0,0.000000,233.692,342.031067
52186,24,52186,193783.0,0.0,0.000000,0.127,344.256700


In [42]:
signal_analysis = features[
    [
        "vehicle_id",
        "track_id",
        "n_signal_loss_valid",
        "distance_km",
        "duration_minutes",
    ]
].copy()

signal_analysis["signal_losses_per_hour"] = (
    signal_analysis["n_signal_loss_valid"]
    / (signal_analysis["duration_minutes"] / 60).where(
        signal_analysis["duration_minutes"] > 0
    )
)

signal_analysis["signal_losses_per_km"] = (
    signal_analysis["n_signal_loss_valid"]
    / signal_analysis["distance_km"].where(
        signal_analysis["distance_km"] > 0
    )
)

display(
    signal_analysis[
        ["signal_losses_per_hour", "signal_losses_per_km"]
    ].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
)

,signal_losses_per_hour,signal_losses_per_km
count,149613.000000,1.479590e+05
mean,1094.139578,3.911783e+04
std,5167.026149,1.020482e+06
min,0.000000,0.000000e+00
25%,0.000000,0.000000e+00
50%,8.709305,3.560608e-01
75%,20.419450,2.173913e+01
90%,353.171431,1.666667e+02
95%,2963.507389,5.782179e+02
99%,33346.554239,8.118477e+04


In [34]:
# Add relevant existing features to the temporary analysis dataframe
extra_cols = [
    "avg_speed_valid",
    "max_speed_valid",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
    "distance",
    "duration_seconds",
    "telemetry_span_seconds",
    "avg_hdop",
    "raw_hdop_mean",
    "raw_hdop_max",
    "d_signal_loss",
    "r_signal_loss_valid",
    "speed_distance_validity",
]

available_cols = [
    col for col in extra_cols
    if col in features.columns
]

signal_analysis = signal_analysis.join(
    features[available_cols].drop(
        columns=[
            col for col in available_cols
            if col in signal_analysis.columns
        ],
        errors="ignore",
    )
)

# Inspect the highest rates, with all relevant parameters
analysis_cols = [
    "vehicle_id",
    "track_id",
    "n_signal_loss_valid",
    "duration_minutes",
    "distance_km",
    "signal_losses_per_hour",
    "signal_losses_per_km",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
    "avg_hdop",
    "raw_hdop_mean",
    "d_signal_loss",
    "r_signal_loss_valid",
]

analysis_cols = [
    col for col in analysis_cols
    if col in signal_analysis.columns
]

print("Highest signal-loss rate per hour")
display(
    signal_analysis.sort_values(
        "signal_losses_per_hour",
        ascending=False,
    )[analysis_cols].head(20)
)

print("Highest signal-loss rate per kilometre")
display(
    signal_analysis.sort_values(
        "signal_losses_per_km",
        ascending=False,
    )[analysis_cols].head(20)
)

Highest signal-loss rate per hour


,vehicle_id,track_id,n_signal_loss_valid,duration_minutes,distance_km,signal_losses_per_hour,signal_losses_per_km,avg_speed_kmh,max_speed_kmh,raw_speed_mean_kmh,avg_hdop,raw_hdop_mean,d_signal_loss,r_signal_loss_valid
148834,155,148834,80.0,0.131283,0.000,36562.142948,NaN,0.000,16.524,10.719797,6.522412,6.487342,0.0,NaN
148839,155,148839,80.0,0.131300,0.000,36557.501904,NaN,0.000,3.600,1.975899,9.794562,9.726582,0.0,NaN
140666,126,140666,80.0,0.131300,0.000,36557.501904,NaN,0.000,0.900,0.457519,9.296662,9.243038,0.0,NaN
147277,149,147277,80.0,0.131317,0.000,36552.862038,NaN,0.000,4.752,4.015139,10.157312,10.144304,0.0,NaN
131598,83,131598,80.0,0.131317,0.000,36552.862038,NaN,0.000,5.184,2.327241,6.083662,6.089873,0.0,NaN
147208,149,147208,80.0,0.131317,0.000,36552.862038,NaN,0.000,1.332,0.826633,9.862787,9.834177,0.0,NaN
50805,23,50805,83.0,0.137633,0.000,36183.095180,NaN,0.000,60.768,53.758976,9.645699,9.650000,0.0,NaN
51212,23,51212,83.0,0.137650,0.000,36178.714130,NaN,0.000,3.384,1.733707,6.018217,6.023171,0.0,NaN
109287,56,109287,83.0,0.137667,0.000,36174.334140,NaN,0.000,1.008,0.360439,7.507928,7.429268,0.0,NaN
148292,153,148292,161.0,0.267617,0.000,36096.406552,NaN,0.000,2.952,0.870525,6.258118,6.242500,0.0,NaN


Highest signal-loss rate per kilometre


,vehicle_id,track_id,n_signal_loss_valid,duration_minutes,distance_km,signal_losses_per_hour,signal_losses_per_km,avg_speed_kmh,max_speed_kmh,raw_speed_mean_kmh,avg_hdop,raw_hdop_mean,d_signal_loss,r_signal_loss_valid
96369,50,96369,95258.0,163.923067,0.001,34866.844040,9.525800e+07,0.0,92.412,60.559467,0.519190,0.519422,0.0,0.0
89125,46,89125,184683.0,336.870033,0.002,32893.932091,9.234150e+07,0.0,95.364,39.089943,3.275453,3.275663,0.0,0.0
144443,138,144443,90649.0,186.929550,0.001,29096.202286,9.064900e+07,0.0,93.636,65.944269,0.923286,0.923046,0.0,0.0
88172,45,88172,170577.0,293.284100,0.002,34896.607078,8.528850e+07,0.0,97.164,64.359447,0.505038,0.505506,0.0,0.0
90421,46,90421,77698.0,137.436200,0.001,33920.320847,7.769800e+07,0.0,90.576,65.402554,0.569405,0.568237,0.0,0.0
605,1,605,153893.0,262.058400,0.002,35234.817888,7.694650e+07,0.0,92.484,54.197642,0.799543,0.799994,0.0,0.0
92929,48,92929,68293.0,117.338833,0.001,34920.919900,6.829300e+07,0.0,95.184,78.489194,0.926251,0.926140,0.0,0.0
89070,46,89070,190215.0,324.432783,0.003,35178.010936,6.340500e+07,0.0,0.792,0.086034,0.555277,0.555904,0.0,0.0
87936,45,87936,125383.0,215.151367,0.002,34965.987512,6.269150e+07,0.0,95.328,81.774400,0.687836,0.688758,0.0,0.0
82820,41,82820,62076.0,108.928250,0.001,34192.782864,6.207600e+07,0.0,91.188,71.862011,0.685902,0.686024,0.0,0.0


In [40]:
features["signal_loss_validity"] = np.where(
    (features["avg_speed_valid"] == 0)
    & (features["distance"] == 0)
    & (features["n_signal_loss_valid"] >= 13),
    "invalid",
    "valid"
)

In [37]:
features["signal_loss_validity"].value_counts()

signal_loss_validity
valid      151307
invalid       291
Name: count, dtype: int64

In [38]:
features.loc[
    features["track_id"] == 148946,
    [
        "vehicle_id",
        "track_id",
        "distance",
        "avg_speed",
        "max_speed",
        "n_signal_loss",
        "d_signal_loss",
        "r_signal_loss",
        "duration_seconds",
        "telemetry_span_seconds",
        "avg_hdop",
        "raw_hdop_mean",
        "raw_hdop_max"
    ]
].to_string(index=False)

' vehicle_id  track_id  distance  avg_speed  max_speed  n_signal_loss  d_signal_loss  r_signal_loss  duration_seconds  telemetry_span_seconds  avg_hdop  raw_hdop_mean  raw_hdop_max\n        156    148946   40444.0      10.56      24.59        37942.0            0.0            0.0          3829.107                 3829.02  1.329319         1.3294           2.4'

In [43]:
features.loc[
    (features["n_signal_loss_valid"] >= 13)
    & (features["distance"] > 0)
    & (features["avg_speed_valid"] > 0),
    [
        "vehicle_id",
        "track_id",
        "distance",
        "avg_speed",
        "n_signal_loss",
        "d_signal_loss",
        "r_signal_loss",
        "duration_seconds",
        "avg_hdop",
        "raw_hdop_mean"
    ]
].sort_values(
    "n_signal_loss_valid",
    ascending=False
).head(20).to_string(index=False)

KeyError: 'n_signal_loss_valid'

In [44]:
features.loc[
    (features["n_signal_loss"] >= 13)
    & (features["distance"] > 0)
    & (features["avg_speed"] > 0),
    [
        "vehicle_id",
        "track_id",
        "distance",
        "avg_speed",
        "n_signal_loss",
        "d_signal_loss",
        "r_signal_loss",
        "duration_seconds",
        "avg_hdop",
        "raw_hdop_mean"
    ]
].sort_values(
    "n_signal_loss",
    ascending=False
).head(20).to_string(index=False)

' vehicle_id  track_id  distance  avg_speed  n_signal_loss  d_signal_loss  r_signal_loss  duration_seconds  avg_hdop  raw_hdop_mean\n         35     74784   20423.0       0.33       532659.0          246.0       0.012045         61784.027  0.950729       0.951498\n         35     74708   42811.0       0.97       391263.0            3.0       0.000070         44175.197  0.693506       0.694492\n         28     63263  213189.0       9.79       211555.0            0.0       0.000000         21785.660  0.816891       0.817117\n         24     52439  249634.0      11.84       205715.0            0.0       0.000000         21083.700  0.947259       0.947726\n         25     54541  185510.0       8.85       203732.0            0.0       0.000000         20959.446  0.676957       0.677285\n         80    130715     578.0       0.03       200072.0            0.0       0.000000         21128.615  0.658859       0.659911\n         24     53439  241239.0      11.73       197234.0            0.0   

In [45]:
# Inspect the raw telemetry file for one extreme-count track
vehicle_id = 28
track_id = 63263

raw = pd.read_csv(
    SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"
)

print("Raw telemetry observations:", len(raw))

print(
    features.loc[
        features["track_id"] == track_id,
        [
            "vehicle_id",
            "track_id",
            "n_signal_loss",
            "duration_seconds",
            "telemetry_span_seconds",
            "distance",
            "d_signal_loss",
            "r_signal_loss"
        ]
    ].to_string(index=False)
)

Raw telemetry observations: 211593
 vehicle_id  track_id  n_signal_loss  duration_seconds  telemetry_span_seconds  distance  d_signal_loss  r_signal_loss
         28     63263       211555.0          21785.66               21785.657  213189.0            0.0            0.0


In [46]:
# Compare reported signal-loss counts with raw telemetry observations
track_ids_to_check = [63263, 52439, 54541, 74784, 74708]

for track_id in track_ids_to_check:
    row = features.loc[features["track_id"] == track_id].iloc[0]

    vehicle_id = int(row["vehicle_id"])
    path = SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"

    raw = pd.read_csv(path)

    print(
        f"Vehicle: {vehicle_id}, "
        f"Track: {track_id}, "
        f"Raw observations: {len(raw):,}, "
        f"Signal-loss count: {row['n_signal_loss']:,.0f}, "
        f"Difference: {len(raw) - row['n_signal_loss']:,.0f}"
    )

Vehicle: 28, Track: 63263, Raw observations: 211,593, Signal-loss count: 211,555, Difference: 38
Vehicle: 24, Track: 52439, Raw observations: 205,797, Signal-loss count: 205,715, Difference: 82
Vehicle: 25, Track: 54541, Raw observations: 203,732, Signal-loss count: 203,732, Difference: 0
Vehicle: 35, Track: 74784, Raw observations: 558,104, Signal-loss count: 532,659, Difference: 25,445
Vehicle: 35, Track: 74708, Raw observations: 399,057, Signal-loss count: 391,263, Difference: 7,794


In [47]:
vehicle_id = 28
track_id = 63263

raw = pd.read_csv(
    SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"
)

print("Columns:", raw.columns.tolist())
display(raw.head(10))
display(raw.tail(10))

print("\nMissing values:")
print(raw.isna().sum())

print("\nDuplicate rows:", raw.duplicated().sum())

Columns: ['epoch', 'speed', 'hdop']


,epoch,speed,hdop
0,1.646218e+09,0.07,1.2
1,1.646218e+09,0.04,1.2
2,1.646218e+09,0.01,1.2
3,1.646218e+09,0.06,1.1
4,1.646218e+09,0.05,1.1
5,1.646218e+09,0.07,1.1
6,1.646218e+09,0.04,1.1
7,1.646218e+09,0.03,1.1
8,1.646218e+09,0.07,1.1
9,1.646218e+09,0.10,1.1


,epoch,speed,hdop
211583,1.646239e+09,0.01,0.9
211584,1.646239e+09,0.02,0.9
211585,1.646239e+09,0.01,0.9
211586,1.646239e+09,0.04,0.9
211587,1.646239e+09,0.02,0.9
211588,1.646239e+09,0.03,0.9
211589,1.646239e+09,0.01,0.9
211590,1.646239e+09,0.03,0.9
211591,1.646239e+09,0.02,0.9
211592,1.646239e+09,0.01,0.9



Missing values:
epoch    0
speed    0
hdop     0
dtype: int64

Duplicate rows: 0


In [48]:
print(raw["speed"].describe(percentiles=[0.5, 0.9, 0.99]))
print("\nSpeed value counts:")
print(raw["speed"].value_counts().head(15))

count    211593.000000
mean          9.629955
std           8.706702
min           0.000000
50%          11.330000
90%          19.450000
99%          19.970000
max          21.330000
Name: speed, dtype: float64

Speed value counts:
speed
0.02     19062
0.03     15582
0.01     15510
0.04     11023
0.05      7104
0.06      4474
0.07      2689
0.00      2150
0.08      1692
19.33     1350
19.30     1298
19.31     1294
19.34     1283
19.32     1266
19.36     1257
Name: count, dtype: int64


In [49]:
vehicle_id = 28
track_id = 63263

raw = pd.read_csv(
    SPEED_ROOT / str(vehicle_id) / f"{track_id}.csv"
).sort_values("epoch").reset_index(drop=True)

# Count samples that fail the documented speed/HDOP criteria
quality_failure = (
    (raw["hdop"] >= 50)
    | (raw["speed"] < 1.388)
    | (raw["speed"] > 75)
)

# Inspect gaps in the raw timestamps
time_diff = raw["epoch"].diff()
median_period = time_diff[time_diff > 0].median()

large_gaps = time_diff >= 3 * median_period

reported_count = features.loc[
    (features["vehicle_id"] == vehicle_id)
    & (features["track_id"] == track_id),
    "n_signal_loss"
].iloc[0]

print(f"Raw observations: {len(raw):,}")
print(f"Reported signal-loss count: {reported_count:,.0f}")
print(f"Samples failing visible quality criteria: {quality_failure.sum():,}")
print(f"Median sampling interval: {median_period:.4f} seconds")
print(f"Large timestamp gaps: {large_gaps.sum():,}")

Raw observations: 211,593
Reported signal-loss count: 211,555
Samples failing visible quality criteria: 86,349
Median sampling interval: 0.1020 seconds
Large timestamp gaps: 68


In [50]:
print(
    "Observations below 1.388 m/s:",
    (raw["speed"] < 1.388).sum()
)

print(
    "Observations at or above 1.388 m/s:",
    (raw["speed"] >= 1.388).sum()
)

print(
    "Observations above 75 m/s:",
    (raw["speed"] > 75).sum()
)

print(
    "Observations with HDOP >= 50:",
    (raw["hdop"] >= 50).sum()
)


Observations below 1.388 m/s: 86349
Observations at or above 1.388 m/s: 125244
Observations above 75 m/s: 0
Observations with HDOP >= 50: 0


In [51]:
# Inspect extreme signal-loss counts among tracks with plausible movement
high_signal_loss_tracks = features.loc[
    (features["n_signal_loss"] >= 100_000)
    & (features["distance"] > 0)
    & (features["avg_speed"] > 0),
    [
        "vehicle_id",
        "track_id",
        "distance",
        "avg_speed",
        "max_speed",
        "n_signal_loss",
        "d_signal_loss",
        "r_signal_loss",
        "duration_seconds",
        "avg_hdop"
    ]
].sort_values(
    "n_signal_loss",
    ascending=False
)

display(high_signal_loss_tracks.to_string(index=False))

' vehicle_id  track_id  distance  avg_speed  max_speed  n_signal_loss  d_signal_loss  r_signal_loss  duration_seconds  avg_hdop\n         35     74784   20423.0       0.33      21.28       532659.0          246.0       0.012045         61784.027  0.950729\n         35     74708   42811.0       0.97      19.89       391263.0            3.0       0.000070         44175.197  0.693506\n         28     63263  213189.0       9.79      21.33       211555.0            0.0       0.000000         21785.660  0.816891\n         24     52439  249634.0      11.84      23.49       205715.0            0.0       0.000000         21083.700  0.947259\n         25     54541  185510.0       8.85      25.09       203732.0            0.0       0.000000         20959.446  0.676957\n         80    130715     578.0       0.03      25.89       200072.0            0.0       0.000000         21128.615  0.658859\n         24     53439  241239.0      11.73      20.70       197234.0            0.0       0.000000     

In [52]:
# Screen for extreme signal-loss counts
high_signal_loss = features.loc[
    features["n_signal_loss"] >= 100_000
].copy()

# High counts with recorded movement
high_signal_loss_moving = high_signal_loss.loc[
    (high_signal_loss["distance"] > 0)
    & (high_signal_loss["avg_speed"] > 0)
].copy()

print("Total tracks:", len(features))
print("Tracks with n_signal_loss >= 100,000:", len(high_signal_loss))
print(
    "High-count tracks with positive distance and speed:",
    len(high_signal_loss_moving)
)
print(
    "High-count tracks with zero distance or non-positive speed:",
    len(high_signal_loss) - len(high_signal_loss_moving)
)

print("\nSignal-loss count distribution among moving tracks:")
print(
    high_signal_loss_moving["n_signal_loss"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

Total tracks: 151598
Tracks with n_signal_loss >= 100,000: 404
High-count tracks with positive distance and speed: 305
High-count tracks with zero distance or non-positive speed: 99

Signal-loss count distribution among moving tracks:
count       305.000000
mean     131939.462295
std       36274.492359
min      100396.000000
25%      111784.000000
50%      125122.000000
75%      142849.000000
90%      165454.400000
95%      181758.600000
99%      205635.680000
max      532659.000000
Name: n_signal_loss, dtype: float64


In [53]:
high_signal_loss_operational = features.loc[
    (features["n_signal_loss"] >= 100_000)
    & (features["distance"] > 1000)
    & (features["avg_speed"] > 0)
].copy()

print(
    "High-count moving tracks in operational dataset:",
    len(high_signal_loss_operational)
)

print(
    "Percentage of operational dataset:",
    round(
        len(high_signal_loss_operational)
        / (features["distance"] > 1000).sum()
        * 100,
        3
    ),
    "%"
)

High-count moving tracks in operational dataset: 210
Percentage of operational dataset: 0.487 %


In [54]:
skewness_columns = [
    "r_signal_loss_valid",
    "distance_km",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
    "duration_minutes"
]

skewness_summary = (
    features[skewness_columns]
    .skew()
    .sort_values(ascending=False)
    .rename("skewness")
    .to_frame()
)

display(skewness_summary)

,skewness
avg_speed_kmh,52.939378
r_signal_loss_valid,10.527364
distance_km,4.284422
duration_minutes,3.434279
raw_speed_mean_kmh,1.834900
max_speed_kmh,1.107065


In [55]:
operational = features.loc[
    features["distance_km"] > 1
].copy()

speed_columns = [
    "vehicle_id",
    "track_id",
    "distance_km",
    "duration_minutes",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
    "speed_distance_validity"
]

print("Operational tracks:", len(operational))

display(
    operational[speed_columns]
    .sort_values("avg_speed_kmh", ascending=False)
    .head(20)
)

Operational tracks: 43125


,vehicle_id,track_id,distance_km,duration_minutes,avg_speed_kmh,max_speed_kmh,raw_speed_mean_kmh,speed_distance_validity
138274,114,138274,277.421,3.756667,4430.844,8.388,1.931023,invalid
49132,22,49132,182.083,2.706517,4036.536,32.940,7.055322,invalid
135017,102,135017,68.080,1.085617,3762.648,24.696,7.042904,invalid
102170,54,102170,17.535,0.280250,3754.152,0.252,0.083494,invalid
93407,48,93407,104.576,2.277983,2754.432,0.324,0.080933,invalid
9707,5,9707,12.865,0.280383,2753.028,0.252,0.093036,invalid
138899,118,138899,117.550,2.644717,2666.844,24.408,3.576502,invalid
91783,47,91783,226.810,5.702400,2386.476,35.640,11.022743,invalid
143352,135,143352,106.570,2.738200,2335.176,48.816,20.892816,invalid
150232,159,150232,88.133,2.312567,2286.612,16.380,3.786467,invalid


In [56]:
review_columns = [
    "r_signal_loss_valid",
    "duration_minutes",
    "raw_speed_mean_kmh",
    "max_speed_kmh"
]

operational = features.loc[
    features["distance_km"] > 1
].copy()

for col in review_columns:
    s = operational[col]

    print(f"\n{'=' * 55}")
    print(f"Column: {col}")
    print(f"Missing: {s.isna().sum():,}")
    print(f"Zero: {(s == 0).sum():,}")
    print(f"Negative: {(s < 0).sum():,}")
    print(f"Positive: {(s > 0).sum():,}")
    print("\nQuantiles:")
    print(
        s.quantile([
            0.00, 0.25, 0.50, 0.75,
            0.90, 0.95, 0.99, 1.00
        ])
    )


Column: r_signal_loss_valid
Missing: 20
Zero: 12,993
Negative: 0
Positive: 30,112

Quantiles:
0.00    0.000000
0.25    0.000000
0.50    0.001887
0.75    0.009080
0.90    0.032883
0.95    0.073351
0.99    0.201998
1.00    0.999830
Name: r_signal_loss_valid, dtype: float64

Column: duration_minutes
Missing: 4
Zero: 0
Negative: 0
Positive: 43,121

Quantiles:
0.00       0.280250
0.25      20.414850
0.50      42.380767
0.75      83.035700
0.90     141.672117
0.95     193.254467
0.99     264.632013
1.00    1029.733783
Name: duration_minutes, dtype: float64

Column: raw_speed_mean_kmh
Missing: 4
Zero: 0
Negative: 0
Positive: 43,121

Quantiles:
0.00      0.063221
0.25     17.869728
0.50     38.783499
0.75     57.245378
0.90     69.211960
0.95     74.084156
0.99     80.234676
1.00    255.226737
Name: raw_speed_mean_kmh, dtype: float64

Column: max_speed_kmh
Missing: 0
Zero: 0
Negative: 0
Positive: 43,125

Quantiles:
0.00      0.252
0.25     67.788
0.50     82.548
0.75     91.080
0.90     94.03

In [57]:
import numpy as np
import pandas as pd

operational = features.loc[
    features["distance_km"] > 1
].copy()

candidate_columns = [
    "avg_speed_kmh",
    "r_signal_loss_valid"
]

for col in candidate_columns:
    q1 = operational[col].quantile(0.25)
    q3 = operational[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    operational[f"{col}_outlier_flag"] = (
        (operational[col] < lower_bound)
        | (operational[col] > upper_bound)
    )

    print(f"\n{col}")
    print(f"Lower bound: {lower_bound:.4f}")
    print(f"Upper bound: {upper_bound:.4f}")
    print(
        "Flagged:",
        operational[f"{col}_outlier_flag"].sum()
    )


avg_speed_kmh
Lower bound: -41.5980
Upper bound: 116.0820
Flagged: 240

r_signal_loss_valid
Lower bound: -0.0136
Upper bound: 0.0227
Flagged: 5191


In [58]:
review_columns = [
    "vehicle_id",
    "track_id",
    "distance_km",
    "duration_minutes",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
    "r_signal_loss_valid"
]

print("Highest average-speed outliers:")
display(
    operational.loc[
        operational["avg_speed_kmh_outlier_flag"],
        review_columns
    ]
    .sort_values("avg_speed_kmh", ascending=False)
    .head(20)
)

print("\nHighest signal-loss-ratio outliers:")
display(
    operational.loc[
        operational["r_signal_loss_valid_outlier_flag"],
        review_columns
    ]
    .sort_values("r_signal_loss_valid", ascending=False)
    .head(20)
)

Highest average-speed outliers:


,vehicle_id,track_id,distance_km,duration_minutes,avg_speed_kmh,max_speed_kmh,raw_speed_mean_kmh,r_signal_loss_valid
138274,114,138274,277.421,3.756667,4430.844,8.388,1.931023,0.000000
49132,22,49132,182.083,2.706517,4036.536,32.940,7.055322,0.005399
135017,102,135017,68.080,1.085617,3762.648,24.696,7.042904,0.001381
102170,54,102170,17.535,0.280250,3754.152,0.252,0.083494,0.002851
93407,48,93407,104.576,2.277983,2754.432,0.324,0.080933,0.000392
9707,5,9707,12.865,0.280383,2753.028,0.252,0.093036,0.000000
138899,118,138899,117.550,2.644717,2666.844,24.408,3.576502,0.000000
91783,47,91783,226.810,5.702400,2386.476,35.640,11.022743,0.006724
143352,135,143352,106.570,2.738200,2335.176,48.816,20.892816,0.000019
150232,159,150232,88.133,2.312567,2286.612,16.380,3.786467,0.000000



Highest signal-loss-ratio outliers:


,vehicle_id,track_id,distance_km,duration_minutes,avg_speed_kmh,max_speed_kmh,raw_speed_mean_kmh,r_signal_loss_valid
2775,2,2775,5.865,2.519133,139.680,305.352,255.226737,0.999830
2807,2,2807,2.449,2.384017,61.632,59.868,2.862720,0.997958
3845,2,3845,4.299,3.994050,64.584,15.696,2.275463,0.996743
64481,30,64481,67.756,155.315250,26.172,69.696,12.249830,0.994923
140440,126,140440,7.234,57.708817,7.524,86.976,16.070010,0.994471
3318,2,3318,2.198,4.470733,29.484,208.188,2.193449,0.994086
3896,2,3896,11.005,2.984500,221.256,209.052,12.960000,0.993548
2644,2,2644,19.292,17.911017,64.620,316.260,55.140904,0.991240
2792,2,2792,5.416,3.907117,83.160,455.940,5.936354,0.988368
2616,2,2616,9.280,12.187333,45.684,212.544,141.458667,0.987500


In [59]:
# Work with the existing operational dataset and IQR flags.

signal_speed_review = operational[
    [
        "track_id",
        "avg_speed_kmh",
        "max_speed_kmh",
        "raw_speed_mean_kmh",
        "r_signal_loss_valid",
        "avg_speed_kmh_outlier_flag",
        "r_signal_loss_valid_outlier_flag"
    ]
].copy()

high_signal_ratio = signal_speed_review[
    signal_speed_review["r_signal_loss_valid_outlier_flag"]
]

print("High signal-loss-ratio tracks:", len(high_signal_ratio))

print(
    "Also flagged for average speed:",
    high_signal_ratio["avg_speed_kmh_outlier_flag"].sum()
)

print(
    "Percentage also flagged for average speed:",
    round(
        high_signal_ratio["avg_speed_kmh_outlier_flag"].mean() * 100,
        2
    ),
    "%"
)

High signal-loss-ratio tracks: 5191
Also flagged for average speed: 35
Percentage also flagged for average speed: 0.67 %


In [60]:
high_signal_ratio = operational.loc[
    operational["r_signal_loss_valid_outlier_flag"]
].copy()

inconsistent_speed = (
    high_signal_ratio["avg_speed_kmh"]
    > high_signal_ratio["max_speed_kmh"]
)

print("High signal-loss-ratio tracks:", len(high_signal_ratio))
print(
    "Tracks where avg_speed_kmh > max_speed_kmh:",
    inconsistent_speed.sum()
)
print(
    "Percentage:",
    round(inconsistent_speed.mean() * 100, 2),
    "%"
)

High signal-loss-ratio tracks: 5191
Tracks where avg_speed_kmh > max_speed_kmh: 61
Percentage: 1.18 %


In [61]:
# Check whether speed columns are duplicate unit conversions.

checks = {
    "avg_speed_valid vs avg_speed_kmh":
        ("avg_speed_valid", "avg_speed_kmh", 3.6),
    "max_speed_valid vs max_speed_kmh":
        ("max_speed_valid", "max_speed_kmh", 3.6),
}

for label, (col1, col2, factor) in checks.items():
    comparison = features[[col1, col2]].dropna()
    difference = (
        comparison[col1] * factor - comparison[col2]
    ).abs()

    print(f"\n{label}")
    print("Rows compared:", len(comparison))
    print("Maximum absolute difference:", difference.max())
    print("Exact matches:", (difference < 1e-6).sum())


avg_speed_valid vs avg_speed_kmh
Rows compared: 151543
Maximum absolute difference: 0.0
Exact matches: 151543

max_speed_valid vs max_speed_kmh
Rows compared: 151549
Maximum absolute difference: 0.0
Exact matches: 151549


In [62]:
col = "d_signal_loss"

print("Summary statistics:")
print(features[col].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nMissing values:", features[col].isna().sum())
print("Negative values:", (features[col] < 0).sum())
print("Zero values:", (features[col] == 0).sum())
print("Positive values:", (features[col] > 0).sum())

print("\nSkewness:", features[col].skew())

Summary statistics:
count    151549.000000
mean        198.464549
std        1391.952990
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
90%         143.000000
95%         706.000000
99%        4188.000000
max       67936.000000
Name: d_signal_loss, dtype: float64

Missing values: 49
Negative values: 0
Zero values: 111918
Positive values: 39631

Skewness: 18.324965887213494


In [63]:
cols = [
    "d_signal_loss",
    "distance_km",
    "r_signal_loss_valid"
]

print(features[cols].corr())

                     d_signal_loss  distance_km  r_signal_loss_valid
d_signal_loss             1.000000     0.393862             0.298758
distance_km               0.393862     1.000000             0.027204
r_signal_loss_valid       0.298758     0.027204             1.000000


In [64]:
operational = features.loc[features["distance_km"] > 1].copy()

print(
    operational[
        ["d_signal_loss", "distance_km", "r_signal_loss_valid"]
    ].corr()
)

                     d_signal_loss  distance_km  r_signal_loss_valid
d_signal_loss             1.000000     0.334099             0.486646
distance_km               0.334099     1.000000            -0.025424
r_signal_loss_valid       0.486646    -0.025424             1.000000


In [65]:
col = "d_signal_loss"

print("Missing:", operational[col].isna().sum())
print("Negative:", (operational[col] < 0).sum())
print("Zero:", (operational[col] == 0).sum())
print("Positive:", (operational[col] > 0).sum())

print("\nDistribution:")
print(
    operational[col].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nSkewness:", operational[col].skew())

Missing: 0
Negative: 0
Zero: 12993
Positive: 30132

Distribution:
count    43125.000000
mean       690.369577
std       2537.689465
min          0.000000
25%          0.000000
50%         35.000000
75%        326.000000
90%       1588.000000
95%       3111.800000
99%      11614.000000
max      67936.000000
Name: d_signal_loss, dtype: float64

Skewness: 10.01744145712759


In [66]:
q1 = operational["d_signal_loss"].quantile(0.25)
q3 = operational["d_signal_loss"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

operational["d_signal_loss_iqr_outlier"] = (
    (operational["d_signal_loss"] < lower_bound) |
    (operational["d_signal_loss"] > upper_bound)
)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Candidate outliers:",
      operational["d_signal_loss_iqr_outlier"].sum())

Q1: 0.0
Q3: 326.0
IQR: 326.0
Lower bound: -489.0
Upper bound: 815.0
Candidate outliers: 6888


In [67]:
cols = [
    "vehicle_id",
    "track_id",
    "distance_km",
    "duration_minutes",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
    "d_signal_loss",
    "r_signal_loss_valid",
]

extreme_tracks = (
    operational.nlargest(20, "d_signal_loss")[cols]
)

print(extreme_tracks.to_string(index=False))

 vehicle_id  track_id  distance_km  duration_minutes  avg_speed_kmh  max_speed_kmh  raw_speed_mean_kmh  d_signal_loss  r_signal_loss_valid
        113    137895      132.151        126.120667         62.856        116.172           64.144388        67936.0             0.514079
         76    129448      170.665        231.244650         44.280         89.316           55.850488        67885.0             0.397768
         30     64481       67.756        155.315250         26.172         69.696           12.249830        67412.0             0.994923
         20     42601      157.038        177.740767         53.028        105.552           64.715685        63720.0             0.405762
        147    146550       70.803        112.923683         37.620         96.408           46.471488        60358.0             0.852478
        135    143468       71.177         87.398650         48.852         88.308           62.089892        59268.0             0.832685
        126    140434      

In [68]:
operational["distance_m"] = operational["distance_km"] * 1000

operational["signal_distance_exceeds_track"] = (
    operational["d_signal_loss"] > operational["distance_m"]
)

print(
    "Tracks where signal-loss distance exceeds total track distance:",
    operational["signal_distance_exceeds_track"].sum()
)

print(
    operational.loc[
        operational["signal_distance_exceeds_track"],
        [
            "vehicle_id",
            "track_id",
            "distance_km",
            "d_signal_loss",
            "r_signal_loss_valid",
        ],
    ].head(20).to_string(index=False)
)

Tracks where signal-loss distance exceeds total track distance: 20
 vehicle_id  track_id  distance_km  d_signal_loss  r_signal_loss_valid
         15     28972        2.115         3528.0                  NaN
         18     39102       13.172        36200.0                  NaN
         22     49759        1.612         1853.0                  NaN
         29     64197        1.902         1916.0                  NaN
         43     84264        2.960         3405.0                  NaN
         44     87274        1.220         3048.0                  NaN
         45     88074        1.075         1240.0                  NaN
         54    103288       29.916        31697.0                  NaN
         54    103882        5.150        20991.0                  NaN
         59    116380        1.422         5185.0                  NaN
         59    116381        1.422         5185.0                  NaN
         59    117023        5.769         9053.0                  NaN
         6

In [69]:
operational["d_signal_loss_validity"] = np.where(
    operational["d_signal_loss"] <= operational["distance_km"] * 1000,
    "valid",
    "invalid"
)

print(operational["d_signal_loss_validity"].value_counts())

d_signal_loss_validity
valid      43105
invalid       20
Name: count, dtype: int64


In [70]:
# Vehicle concentration among signal-distance inconsistencies
invalid_signal_distance = operational[
    operational["d_signal_loss_validity"] == "invalid"
]

print("Invalid records by vehicle:")
print(
    invalid_signal_distance["vehicle_id"]
    .value_counts()
    .sort_values(ascending=False)
)

# Check repeated combinations of total distance and signal-loss distance
print("\nRepeated distance combinations:")
print(
    operational.groupby(
        ["distance_km", "d_signal_loss"]
    ).size()
    .loc[lambda s: s > 1]
    .sort_values(ascending=False)
    .head(20)
)

Invalid records by vehicle:
vehicle_id
59     3
54     2
61     2
77     2
15     1
18     1
22     1
29     1
43     1
44     1
45     1
93     1
116    1
144    1
151    1
Name: count, dtype: int64

Repeated distance combinations:
distance_km  d_signal_loss
1.075        0.0              15
1.014        0.0              15
1.030        0.0              14
1.025        0.0              14
1.045        0.0              13
1.055        0.0              13
1.056        0.0              13
1.009        0.0              12
1.062        0.0              12
1.023        0.0              12
1.011        0.0              12
1.001        0.0              12
1.042        0.0              12
1.022        0.0              11
1.028        0.0              11
1.059        0.0              10
1.016        0.0              10
1.038        0.0              10
1.019        0.0              10
1.178        0.0              10
dtype: int64


In [71]:
print(
    operational["d_signal_loss_validity"]
    .value_counts()
    .to_string()
)

print(
    "\nInvalid percentage:",
    (
        operational["d_signal_loss_validity"].eq("invalid").mean()
        * 100
    )
)

print(
    "\nSkewness among consistency-valid records:",
    operational.loc[
        operational["d_signal_loss_validity"] == "valid",
        "d_signal_loss"
    ].skew()
)

d_signal_loss_validity
valid      43105
invalid       20

Invalid percentage: 0.0463768115942029

Skewness among consistency-valid records: 10.084868300495327


In [72]:

from pathlib import Path

# Confirm that the engineered DataFrame exists
if "features" not in globals():
    raise NameError(
        "The DataFrame 'features' was not found. "
        "Run the feature-engineering cells first."
    )

# Set the output directory relative to the current working directory.
# If your notebook is in the notebooks/ folder, the project root is its parent.
cwd = Path.cwd()

if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd

output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Output file for the PCA notebook
engineered_path = output_dir / "engineered_tracks.csv"

# Save the full engineered dataset without changing the DataFrame
features.to_csv(engineered_path, index=False)

# Verify the saved file
print("=" * 50)
print("ENGINEERED DATASET SAVED")
print("=" * 50)
print(f"Project root: {project_root}")
print(f"File path:    {engineered_path.resolve()}")
print(f"Rows:         {features.shape[0]:,}")
print(f"Columns:      {features.shape[1]}")
print(f"File exists:  {engineered_path.exists()}")
print(f"File size:    {engineered_path.stat().st_size / (1024**2):.2f} MB")
print("=" * 50)


ENGINEERED DATASET SAVED
Project root: /Users/janibhargesh/Documents/ML-Project
File path:    /Users/janibhargesh/Documents/ML-Project/data/processed/engineered_tracks.csv
Rows:         151,598
Columns:      63
File exists:  True
File size:    78.63 MB
